# Import libraries

In [1]:
import pandas as pd # handle data (tables)
from sklearn.model_selection import train_test_split # split data into train & test
from sklearn.linear_model import LogisticRegression # ML algorithm
from sklearn.metrics import accuracy_score # measure performance
from sklearn.linear_model import SGDClassifier # to visualize training process
import joblib # to save the best model
import os # to check if model file exists

# Create Dataset

In [2]:
boardings = pd.read_csv("boarding_data.csv")

boardings.head(500)

,id,number_of_rooms,distance_km,rating,parking,security,water,laundry,bus_route,quiet,ac,wifi,bathroom,kitchen,study_desk,entrance
0,1,2,1.11,3.2,0,0,1,0,1,1,1,0,1,1,0,0
1,2,2,5.86,3.8,0,1,0,1,1,1,1,1,0,1,1,0
2,3,1,19.09,2.9,1,1,0,0,0,1,1,0,0,1,0,1
3,4,6,15.29,4.5,1,1,1,0,1,0,1,1,1,1,1,1
4,5,4,17.76,2.6,0,1,1,0,0,0,1,0,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,496,2,20.64,4.8,1,0,0,1,0,1,1,1,1,0,0,1
496,497,1,17.33,4.0,0,1,1,0,0,1,1,1,1,1,0,1
497,498,5,16.15,3.0,0,0,1,0,1,1,1,1,1,0,0,0
498,499,3,7.20,2.5,1,1,1,0,0,1,1,1,1,0,0,1


In [3]:
users = pd.read_csv("users.csv")

users.head(100)

,user_id,preferred_rooms,max_distance,requested_facilities
0,U1,1,3.3,kitchen
1,U2,1,7.4,"bathroom,entrance,security,wifi,bus_route"
2,U3,1,4.4,"bus_route,entrance,bathroom,security"
3,U4,1,9.1,"kitchen,parking,ac,quiet"
4,U5,1,3.0,bus_route
...,...,...,...,...
95,U96,1,5.3,"study_desk,parking,laundry,kitchen,entrance,quiet"
96,U97,4,7.5,"water,quiet,laundry,bus_route"
97,U98,4,8.3,"laundry,study_desk,bathroom,bus_route,water"
98,U99,2,3.6,"water,laundry,ac,bathroom,study_desk,quiet"


In [4]:
interactions = pd.read_csv("interaction.csv")

interactions.head(500)

,user_id,boarding_id,label
0,U1,1.0,1
1,U1,2.0,0
2,U1,3.0,0
3,U1,4.0,0
4,U1,5.0,0
...,...,...,...
495,U1,496.0,0
496,U1,497.0,0
497,U1,498.0,0
498,U1,499.0,0


# Convert requested facilities to list

In [5]:
users["requested_facilities"] = users["requested_facilities"].apply(
    lambda x: [f.strip() for f in x.split(",")]
)

print(users)

   user_id  preferred_rooms  max_distance  \
0       U1                1           3.3   
1       U2                1           7.4   
2       U3                1           4.4   
3       U4                1           9.1   
4       U5                1           3.0   
..     ...              ...           ...   
95     U96                1           5.3   
96     U97                4           7.5   
97     U98                4           8.3   
98     U99                2           3.6   
99    U100                1           5.8   

                                 requested_facilities  
0                                           [kitchen]  
1     [bathroom, entrance, security, wifi, bus_route]  
2           [bus_route, entrance, bathroom, security]  
3                       [kitchen, parking, ac, quiet]  
4                                         [bus_route]  
..                                                ...  
95  [study_desk, parking, laundry, kitchen, entran...  
96         

# Merge datasets

In [6]:
data = interactions.merge(users, on="user_id") \
                   .merge(boardings, left_on="boarding_id", right_on="id")

data.head(1000)

,user_id,boarding_id,label,preferred_rooms,max_distance,requested_facilities,id,number_of_rooms,distance_km,rating,...,water,laundry,bus_route,quiet,ac,wifi,bathroom,kitchen,study_desk,entrance
0,U1,1.0,1,1,3.3,[kitchen],1,2,1.11,3.2,...,1,0,1,1,1,0,1,1,0,0
1,U1,2.0,0,1,3.3,[kitchen],2,2,5.86,3.8,...,0,1,1,1,1,1,0,1,1,0
2,U1,3.0,0,1,3.3,[kitchen],3,1,19.09,2.9,...,0,0,0,1,1,0,0,1,0,1
3,U1,4.0,0,1,3.3,[kitchen],4,6,15.29,4.5,...,1,0,1,0,1,1,1,1,1,1
4,U1,5.0,0,1,3.3,[kitchen],5,4,17.76,2.6,...,1,0,0,0,1,0,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,U2,496.0,0,1,7.4,"[bathroom, entrance, security, wifi, bus_route]",496,2,20.64,4.8,...,0,1,0,1,1,1,1,0,0,1
996,U2,497.0,0,1,7.4,"[bathroom, entrance, security, wifi, bus_route]",497,1,17.33,4.0,...,1,0,0,1,1,1,1,1,0,1
997,U2,498.0,0,1,7.4,"[bathroom, entrance, security, wifi, bus_route]",498,5,16.15,3.0,...,1,0,1,1,1,1,1,0,0,0
998,U2,499.0,1,1,7.4,"[bathroom, entrance, security, wifi, bus_route]",499,3,7.20,2.5,...,1,0,0,1,1,1,1,0,0,1


# Feature engineering

In [7]:
def create_features(row):

    # Room match
    room_match = 1 if row["number_of_rooms"] >= row["preferred_rooms"] else 0

    # Distance score (clamped)
    distance_score = max(0, 1 - (row["distance_km"] / max(row["max_distance"], 1)))

    # Rating score
    rating_score = row["rating"]

    # Facility score (DYNAMIC)
    requested = row["requested_facilities"]

    match_count = sum(1 for f in requested if row.get(f, 0) == 1)

    facility_score = match_count / len(requested) if requested else 0

    return pd.Series([room_match, distance_score, rating_score, facility_score])

# Apply features

In [8]:
data[["room_match", "distance_score", "rating_score", "facility_score"]] = data.apply(create_features, axis=1)

data.head(1000)

,user_id,boarding_id,label,preferred_rooms,max_distance,requested_facilities,id,number_of_rooms,distance_km,rating,...,ac,wifi,bathroom,kitchen,study_desk,entrance,room_match,distance_score,rating_score,facility_score
0,U1,1.0,1,1,3.3,[kitchen],1,2,1.11,3.2,...,1,0,1,1,0,0,1.0,0.663636,3.2,1.0
1,U1,2.0,0,1,3.3,[kitchen],2,2,5.86,3.8,...,1,1,0,1,1,0,1.0,0.000000,3.8,1.0
2,U1,3.0,0,1,3.3,[kitchen],3,1,19.09,2.9,...,1,0,0,1,0,1,1.0,0.000000,2.9,1.0
3,U1,4.0,0,1,3.3,[kitchen],4,6,15.29,4.5,...,1,1,1,1,1,1,1.0,0.000000,4.5,1.0
4,U1,5.0,0,1,3.3,[kitchen],5,4,17.76,2.6,...,1,0,1,1,1,0,1.0,0.000000,2.6,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,U2,496.0,0,1,7.4,"[bathroom, entrance, security, wifi, bus_route]",496,2,20.64,4.8,...,1,1,1,0,0,1,1.0,0.000000,4.8,0.6
996,U2,497.0,0,1,7.4,"[bathroom, entrance, security, wifi, bus_route]",497,1,17.33,4.0,...,1,1,1,1,0,1,1.0,0.000000,4.0,0.8
997,U2,498.0,0,1,7.4,"[bathroom, entrance, security, wifi, bus_route]",498,5,16.15,3.0,...,1,1,1,0,0,0,1.0,0.000000,3.0,0.6
998,U2,499.0,1,1,7.4,"[bathroom, entrance, security, wifi, bus_route]",499,3,7.20,2.5,...,1,1,1,0,0,1,1.0,0.027027,2.5,0.8


# Final dataset

In [9]:
X = data[["room_match", "distance_score", "rating_score", "facility_score"]]
y = data["label"]

X.head(1000)

,room_match,distance_score,rating_score,facility_score
0,1.0,0.663636,3.2,1.0
1,1.0,0.000000,3.8,1.0
2,1.0,0.000000,2.9,1.0
3,1.0,0.000000,4.5,1.0
4,1.0,0.000000,2.6,1.0
...,...,...,...,...
995,1.0,0.000000,4.8,0.6
996,1.0,0.000000,4.0,0.8
997,1.0,0.000000,3.0,0.6
998,1.0,0.027027,2.5,0.8


In [10]:
y.head(1000)

0      1
1      0
2      0
3      0
4      0
      ..
995    0
996    0
997    0
998    1
999    0
Name: label, Length: 1000, dtype: int64

# Train-Test Split

👉 You said:

Split data → teach model + test model

That’s correct — but the real reason is:

👉 To check if the model can handle new, unseen data

### 🎯 The real problem (without split)

Let’s say you have this data:

- rooms = 2
- distance = 1.5
- rating = 4.5
- chosen = 1
----------------------------
- rooms = 1
- distance = 5
- rating = 3.0
- chosen = 0

❌ If you train AND test on same data:

Model sees:

2 rooms, 1.5km → chosen = 1

Later you ask:

2 rooms, 1.5km → ?

👉 Model says:

1 (correct)
🤔 But why?

👉 Because it already saw this exact data

**👉 It didn’t learn — it memorized**

🔥 This is called:

**👉 Overfitting (memorization)**

👉 You said:

## ✅ With Train-Test Split

Now split data:

### Training data (Teach model):
- rooms = 2
- distance = 1.5
- rating = 4.5
- chosen = 1

### Testing data (Check model):
- rooms = 1
- distance = 5
- rating = 3.0
- chosen = 0

Now what happens?

👉 Model learns from training data

Then you test:

1 room, 5km → ?

👉 This is new data for the model

### 🧠 Why this is powerful

Now model must:
- understand patterns
- not just remember
## 💡 Real-world analogy
### ❌ Without split

Teacher gives:
lesson
AND exam questions

**👉 You memorize → 100% marks 😄**

# ✅ With split

Teacher gives:lessons

Exam:new questions

👉 Now:

- you must understand
- not memorize

### 🎯 Key idea

**👉 ML is NOT about remembering**

**👉 ML is about generalizing**

### 🔍 Important term

🧠 Generalization

- Ability to perform well on new unseen data

In [11]:
# split data

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# test_size=0.2 -> 20% of data from the dataset will be used for testing
# random_state=42 -> “use the same random split every time”, it's a seed number
# X_train -> features for training
# X_test -> features for testing
# y_train -> correct answers for training
# y_test -> correct answers for testing

print(X_train)
print(X_test)

       room_match  distance_score  rating_score  facility_score
39087         1.0        0.000000           4.6             1.0
30893         1.0        0.093878           4.9             0.2
45278         1.0        0.000000           3.9             1.0
16398         1.0        0.000000           3.9             0.4
13653         1.0        0.000000           3.4             1.0
...           ...             ...           ...             ...
11284         0.0        0.000000           3.6             1.0
44732         1.0        0.000000           4.0             1.0
38158         1.0        0.000000           3.7             0.5
860           1.0        0.000000           2.9             1.0
15795         1.0        0.131250           3.7             0.0

[40000 rows x 4 columns]
       room_match  distance_score  rating_score  facility_score
33553         1.0        0.000000           4.3        0.666667
9427          1.0        0.565116           4.4        1.000000
199           

# Train the model

Training = the model learns patterns from data

In [12]:
model = LogisticRegression(max_iter=2000,C=1.0,random_state=42)
# model = SGDClassifier(
#     loss="log_loss",   # logistic regression behavior
#     max_iter=1,        # only 100 iteration per fit (we control loop)
#     warm_start=True,   # keep learning (don't reset weights)
#     random_state=42
# )

# # 👉 You are:

# # creating a model object
# # choosing the algorithm: Logistic Regression

# # 👉 At this point:

# # Model knows NOTHING ❌

In [13]:
model.fit(X_train, y_train)

# # training loop
# best_acc = 0
# best_iteration = 0

# for i in range(150):
#     model.fit(X_train, y_train)
    
#     y_pred = model.predict(X_test)
#     acc = accuracy_score(y_test, y_pred)
    
#     print(f"Iteration {i+1} - Accuracy: {acc:.4f}")
    
#     if acc > best_acc:
#         best_acc = acc
#         best_iteration = i + 1
        
#         # save best model
#         joblib.dump(model, "best_model.pkl")

# print("\nBest Accuracy:", best_acc)
# print("Best Iteration:", best_iteration)

# # 👉 This is where learning happens

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


## 🧠 What happens inside .fit() (VERY IMPORTANT)
### 🧩 Step 1: Model looks at input

X_train:
rooms, distance, wifi, rating

### 🧩 Step 2: Model tries to predict

It guesses:

chosen = 0 or 1

### 🧩 Step 3: Compare with real answer

Prediction vs Actual (y_train)

### 🧩 Step 4: Calculate error

Example:

- Predicted: 0
- Actual:    1

→ Error ❌

### 🧩 Step 5: Adjust weights

Model updates importance of features:

- distance → maybe too strong
- rating → increase importance
  
### 🧩 Step 6: Repeat MANY times

👉 This loop happens many times:

Predict → Error → Adjust → Repeat

### 🤖 What Logistic Regression is learning

It learns something like:

### score = (rooms × w1) + (distance × w2) + (rating × w3) + ...

👉 These w1, w2 are weights

Then converts score → probability.

$\sigma(z) = \frac{1}{1 + e^{-z}}$

### 🧠 Meaning of this formula
Output is always between 0 and 1
Represents probability
- 💡 Example
- Input:
rooms = 2
distance = 2km
wifi = 1
rating = 4

Model calculates:

Probability = 0.82

👉 Meaning:

82% chance user will choose it

🎯 What you get after training

After .fit():

Model now KNOWS patterns ✅

🔍 Internally model stores:

weights (importance of each feature)

bias (base value)

⚠️ Important

👉 Training uses ONLY:

X_train, y_train

👉 It does NOT see test data

💡 Simple analogy

Think like:

You give examples to a student

Student learns rules

Then applies to new questions

### 🎯 One-line understanding

.fit() = “learn from training data by adjusting weights to reduce error”

# Test the model

In [14]:
predictions = model.predict(X_test)

# Compare the predictions with given lables and calculate the accuracy

In [15]:
accuracy = accuracy_score(y_test, predictions)
print("Test Accuracy:", accuracy)

Test Accuracy: 0.9417


# Save the model

In [16]:
os.makedirs("../model", exist_ok=True)
joblib.dump(model, "../model/boarding_model.pkl")

print("Model trained & saved!")

Model trained & saved!


# Test with new data

In [17]:
from httpx import head

# Load model
model = joblib.load("../model/boarding_model.pkl")

# User input (NEW FORMAT)
user = {
    "rooms": 2,
    "distance": 5,
    "facilities": ["wifi", "parking", "ac"]
}

# Sample boardings (UPDATED STRUCTURE)
boardings = [
    {
        "id": 1,
        "number_of_rooms": 2,
        "distance_km": 1.5,
        "rating": 4.6,
        "wifi": 1,
        "parking": 1,
        "ac": 1,
        "study_desk": 1
    },
    {
        "id": 2,
        "number_of_rooms": 1,
        "distance_km": 5,
        "rating": 3.0,
        "wifi": 0,
        "parking": 0,
        "ac": 0,
        "study_desk": 0
    },
    {
        "id": 3,
        "number_of_rooms": 3,
        "distance_km": 1,
        "rating": 4.8,
        "wifi": 1,
        "parking": 1,
        "ac": 1,
        "study_desk": 1
    }
]

# ---------------- FEATURE FUNCTIONS ----------------

def calculate_facility_score(requested, boarding):
    match = sum(1 for f in requested if boarding.get(f, 0) == 1)
    return match / len(requested) if requested else 0


def create_features(user, boarding):

    room_match = 1 if boarding["number_of_rooms"] >= user["rooms"] else 0

    distance_score = max(0, 1 - (boarding["distance_km"] / max(user["distance"], 1)))

    rating_score = boarding["rating"] / 5

    facility_score = calculate_facility_score(user["facilities"], boarding)

    return [room_match, distance_score, rating_score, facility_score]


# ---------------- BUILD FEATURE MATRIX ----------------

X = []

for b in boardings:
    X.append(create_features(user, b))

X = pd.DataFrame(X, columns=[
    "room_match",
    "distance_score",
    "rating_score",
    "facility_score"
])

print(X)

# ---------------- PREDICT ----------------

probs = model.predict_proba(X)

# Attach scores
for i, b in enumerate(boardings):
    b["score"] = float(probs[i][1])

# Sort
boardings.sort(key=lambda x: x["score"], reverse=True)

# Display
for b in boardings:
    print(b)

   room_match  distance_score  rating_score  facility_score
0           1             0.7          0.92             1.0
1           0             0.0          0.60             0.0
2           1             0.8          0.96             1.0
{'id': 3, 'number_of_rooms': 3, 'distance_km': 1, 'rating': 4.8, 'wifi': 1, 'parking': 1, 'ac': 1, 'study_desk': 1, 'score': 0.9984280620543432}
{'id': 1, 'number_of_rooms': 2, 'distance_km': 1.5, 'rating': 4.6, 'wifi': 1, 'parking': 1, 'ac': 1, 'study_desk': 1, 'score': 0.9953672445658714}
{'id': 2, 'number_of_rooms': 1, 'distance_km': 5, 'rating': 3.0, 'wifi': 0, 'parking': 0, 'ac': 0, 'study_desk': 0, 'score': 6.1013235549425e-08}
